# Notebook de KPIs financeiros
Este notebook calcula e valida os principais indicadores financeiros do projeto

In [1]:
import pandas as pd
import numpy as np
import re

In [3]:
df = pd.read_csv(
    '../staging/dados normalizados.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)

print(df.shape)
print(df.dtypes.head(10))

(14400, 72)
Ano                                             int64
Cenario                                         int64
Taxa                                              str
BAL - Total do Ativo                              str
BAL - Ativo Circulante                            str
BAL - Disponível                                  str
BAL - Contas a Receber - SWAP                     str
BAL - Contas a Receber - Partes Relacionadas      str
BAL - Contas a Receber - Clientes                 str
BAL - Estoques Diversos                           str
dtype: object


In [5]:
def anal_valor_contabil(valor):

    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()

    num_negativo = texto.startswith('(') and texto.endswith(')')
    if num_negativo:
        texto = texto[1:-1]

    texto = texto.replace('.','')
    texto = texto.replace(',','.')

    try:
        numero = float(texto)
    except ValueError:
        return np.nan

    return -numero if num_negativo else numero

In [7]:
#teste
print(anal_valor_contabil('10.631.127.075,26'))  
print(anal_valor_contabil('(103.969.289,50)'))   
print(anal_valor_contabil(np.nan))

10631127075.26
-103969289.5
nan


In [ ]:
colunas_convert = [c for c in df.columns if c not in ('Ano','Cenario')]

for coluna in colunas_convert:
    df[coluna] = df[coluna].apply(anal_valor_contabil)

print(df.dtypes.value_counts())

float64    70
int64       2
Name: count, dtype: int64


array([           nan, 2.06973818e+11, 2.22063650e+11, ...,
       4.18479707e+11, 4.18603102e+11, 4.18727572e+11], shape=(8842,))

In [12]:
#nulos
nulos_coluna = df.isna().sum()
colunas_com_nulo = nulos_coluna[nulos_coluna > 0].sort_values(ascending=False)
print(colunas_com_nulo)

BAL  - Dividendos Antecipados                   3600
DRE - Receitas Financeiras                      2112
FLU - Entradas                                  1294
BAL - Estoques Diversos                         1200
BAL - Contas a Receber - SWAP                   1200
BAL - Contas a Receber - Partes Relacionadas    1200
BAL - Diferido                                  1200
BAL - Amortização Acumulada                     1200
BAL - Contas a Pagar - Parte Relacionada        1200
BAL - Fornecedores                              1200
BAL - Tributos a pagar                          1200
BAL - Créditos Tributários                      1200
BAL - Outorga da Concessão                      1200
BAL - Outros Créditos LP                        1200
BAL  - Capital Social                           1200
BAL - Obrigações com o Poder Concedente         1200
BAL - Encargos Sociais e Trabalhistas           1200
BAL - Outros Débitos                            1200
BAL  - Prov para Contingências                

In [13]:
nulos_por_ano = df.groupby('Ano')[colunas_com_nulo.index.tolist()].apply(lambda x: x.isna().sum())
print(nulos_por_ano)

     BAL  - Dividendos Antecipados  DRE - Receitas Financeiras  \
Ano                                                              
1                             1200                         176   
2                             1200                         176   
3                                0                         176   
4                                0                         176   
5                                0                         176   
6                                0                         176   
7                                0                         176   
8                                0                         176   
9                                0                         176   
10                               0                         176   
11                               0                         176   
12                            1200                         176   

     FLU - Entradas  BAL - Estoques Diversos  BAL - Contas a Receber - SWAP

In [21]:
diferenca_balanco = df['BAL - Total do Ativo'] + df['BAL - Total do Passivo']

df['fecha_balanco'] = np.isclose(diferenca_balanco, 0, atol=0.05)

print(f"Linhas com balanço fechando corretamente: {df['fecha_balanco'].sum()} de {len(df)}")
print(f"Linhas com inconsistência: {(~df['fecha_balanco']).sum()}")

print(diferenca_balanco[~df['fecha_balanco']].describe())

Linhas com balanço fechando corretamente: 0 de 14400
Linhas com inconsistência: 14400
count    1.440000e+04
mean     4.990551e+08
std      3.240153e+11
min     -9.814220e+11
25%      2.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      9.834193e+11
dtype: float64


In [22]:
soma_ativo_categorias = (
    df['BAL - Ativo Circulante']
    + df['BAL - Realizável a Longo Prazo']
    + df['BAL - Permanente']
)

diferenca_ativo = soma_ativo_categorias - df['BAL - Total do Ativo']
df['ativo_consistente'] = np.isclose(diferenca_ativo, 0, atol=0.05)

print(f"Ativo consistente: {df['ativo_consistente'].sum()} de {len(df)}")

Ativo consistente: 6373 de 14400


In [23]:
resultado_operacional_calculado = (
    df['DRE - Receita']
    + df['DRE - Tributos']
    + df['DRE - Custos']
    + df['DRE - Depreciação e Amortização']
)

diferenca_ro = resultado_operacional_calculado - df['DRE - Resultado Operacional']
df['resultado_operacional_confere'] = np.isclose(diferenca_ro, 0, atol=0.05)

print(df['resultado_operacional_confere'].value_counts())

resultado_operacional_confere
False    9521
True     4879
Name: count, dtype: int64


In [25]:
ebitda_calculado = df['DRE - Resultado Operacional'] - df['DRE - Depreciação e Amortização']
residuo_ebitda = df['DRE - EBITDA'] - ebitda_calculado

print("Estatística do resíduo entre EBITDA reportado e EBITDA calculado pela fórmula clássica:")
print(residuo_ebitda.describe())

print(residuo_ebitda.corr(df['BAL - Provisão Manutenção']))

Estatística do resíduo entre EBITDA reportado e EBITDA calculado pela fórmula clássica:
count    1.440000e+04
mean     4.576723e+10
std      2.227348e+11
min     -9.004999e+11
25%      3.176481e+10
50%      3.974957e+10
75%      5.371813e+10
max      1.123933e+12
dtype: float64
-0.055165960561581115


In [26]:
diferenca_caixa = df['FLU - Saldo Final'] - df['BAL - Disponível']
df['caixa_confere'] = np.isclose(diferenca_caixa, 0, atol=0.05)

print(f"Linhas em que o Fluxo de Caixa concilia com o Balanço: {df['caixa_confere'].sum()} de {len(df)}")
print(diferenca_caixa[~df['caixa_confere']].describe())

Linhas em que o Fluxo de Caixa concilia com o Balanço: 14398 de 14400
count    2.000000
mean     0.000000
std      1.414214
min     -1.000000
25%     -0.500000
50%      0.000000
75%      0.500000
max      1.000000
dtype: float64


In [27]:
geracao_calculada = df['FLU - Saldo Final'] - df['FLU - Saldo Inicial']
diferenca_geracao = geracao_calculada - df['FLU - Geração de Caixa']
df['geracao_caixa_confere'] = np.isclose(diferenca_geracao, 0, atol=0.05)

print(df['geracao_caixa_confere'].value_counts())

geracao_caixa_confere
True     7562
False    6838
Name: count, dtype: int64


In [29]:
relatorio_validacao = pd.DataFrame({
    'checagem': [
        'Balanço fecha (Ativo + Passivo = 0)',
        'Ativo = Circulante + Realizável LP + Permanente',
        'Resultado Operacional = Receita+Tributos+Custos+D&A',
        'Fluxo de Caixa concilia com Disponível do Balanço',
        'Geração de Caixa = Saldo Final - Saldo Inicial',
    ],
    'linhas_ok': [
        df['fecha_balanco'].sum(),
        df['ativo_consistente'].sum(),
        df['resultado_operacional_confere'].sum(),
        df['caixa_confere'].sum(),
        df['geracao_caixa_confere'].sum(),
    ],
    'total_linhas': len(df),
})
relatorio_validacao['percentual_ok'] = (relatorio_validacao['linhas_ok'] / relatorio_validacao['total_linhas'] * 100).round(2)
print(relatorio_validacao)

                                            checagem  linhas_ok  total_linhas  \
0                Balanço fecha (Ativo + Passivo = 0)          0         14400   
1    Ativo = Circulante + Realizável LP + Permanente       6373         14400   
2  Resultado Operacional = Receita+Tributos+Custo...       4879         14400   
3  Fluxo de Caixa concilia com Disponível do Balanço      14398         14400   
4     Geração de Caixa = Saldo Final - Saldo Inicial       7562         14400   

   percentual_ok  
0           0.00  
1          44.26  
2          33.88  
3          99.99  
4          52.51  


In [30]:
df['margem_ebitda'] = df['DRE - EBITDA'] / df['DRE - Receita']
df['margem_liquida'] = df['DRE - Resultado Líquido'] / df['DRE - Receita']

In [31]:
df['receita_valida'] = df['DRE - Receita'].abs() > 1

margens_invalidas = (~df['receita_valida']).sum()
print(f"Linhas com receita nula/insignificante (margem não interpretável): {margens_invalidas}")

df.loc[~df['receita_valida'], ['margem_ebitda', 'margem_liquida']] = np.nan

Linhas com receita nula/insignificante (margem não interpretável): 0


In [32]:
df['variacao_caixa_percentual'] = df['FLU - Geração de Caixa'] / df['FLU - Saldo Inicial'].replace(0, np.nan)

In [33]:
resumo_por_ano = df.groupby('Ano').agg(
    margem_ebitda_media=('margem_ebitda', 'mean'),
    margem_ebitda_mediana=('margem_ebitda', 'median'),
    margem_ebitda_p10=('margem_ebitda', lambda x: x.quantile(0.10)),
    margem_ebitda_p90=('margem_ebitda', lambda x: x.quantile(0.90)),
    margem_liquida_media=('margem_liquida', 'mean'),
    geracao_caixa_media=('FLU - Geração de Caixa', 'mean'),
    geracao_caixa_mediana=('FLU - Geração de Caixa', 'median'),
).reset_index()

print(resumo_por_ano)

    Ano  margem_ebitda_media  margem_ebitda_mediana  margem_ebitda_p10  \
0     1             1.270678               0.771328           0.770159   
1     2             1.278112               0.757962           0.756985   
2     3             1.295232               0.783141           0.780151   
3     4             1.330205               0.793861           0.790598   
4     5             1.308729               0.799835           0.795664   
5     6             1.307489               0.805515           0.801190   
6     7             1.391574               0.811138           0.806133   
7     8             1.433105               0.814781           0.810272   
8     9             1.252917               0.818145           0.813790   
9    10             1.417046               0.822863           0.818565   
10   11             1.340383               0.823822           0.818939   
11   12             1.621868               0.956504           0.948032   

    margem_ebitda_p90  margem_liquida

In [34]:
ano_final = df[df['Ano'] == 12].copy()

print(f"Total de cenários no Ano 12 (encerramento): {ano_final['Cenario'].nunique()}")

percentual_caixa_negativo = (ano_final['BAL - Disponível'] < 0).mean() * 100
print(f"Percentual de cenários com caixa negativo no encerramento: {percentual_caixa_negativo:.1f}%")

percentual_pl_negativo = (ano_final['BAL  - Patrimônio Líquido'] < 0).mean() * 100
print(f"Percentual de cenários com Patrimônio Líquido negativo no encerramento: {percentual_pl_negativo:.1f}%")

Total de cenários no Ano 12 (encerramento): 1200
Percentual de cenários com caixa negativo no encerramento: 0.1%
Percentual de cenários com Patrimônio Líquido negativo no encerramento: 0.0%


In [35]:
df_com_kpis = df.assign(
    margem_ebitda=lambda x: x['DRE - EBITDA'] / x['DRE - Receita'],
    margem_liquida=lambda x: x['DRE - Resultado Líquido'] / x['DRE - Receita'],
    variacao_caixa=lambda x: x['FLU - Geração de Caixa'],
)

In [36]:
df['margem_ebitda_media_do_ano'] = df.groupby('Ano')['margem_ebitda'].transform('mean')

df['desvio_margem_vs_media_ano'] = df['margem_ebitda'] - df['margem_ebitda_media_do_ano']

cenarios_abaixo_da_media = df[df['desvio_margem_vs_media_ano'] < -0.05]
print(f"Linhas (Ano, Cenário) com margem EBITDA 5pp abaixo da média do ano: {len(cenarios_abaixo_da_media)}")

Linhas (Ano, Cenário) com margem EBITDA 5pp abaixo da média do ano: 13222
